# Step 54 — Antelope CSS3.0 catalogue and optional MiniSEED wfdisc

This self-contained export notebook runs after step 50. Step 52 remains the independent SEISAN REA exporter. All Antelope-specific code is here; step 50 no longer creates an Antelope database.

## Independent export choices

- **Catalogue metadata:** event, real origin, arrival, assoc, netmag, remark and identifier tables, preserving W/P/H classes and original resource IDs. Unlocated events get no fabricated hypocenter; their arrival/event membership is retained in the JSON sidecar.
- **Optional waveform indexing:** invoke Antelope `miniseed2db` on the already exported MiniSEED files in the configured SEISAN WAV database. This wraps existing recordings in wfdisc (and any ID-mapping tables the native tool creates), without converting or copying waveform samples. The native indexer determines datatype/record offsets; this notebook does not guess offsets for multi-trace or compressed files.

- **Optional network-history export:** load the step-15 StationXML and use ObsPy `Inventory.write(format="CSS")`. Output is staged separately from the catalogue/wfdisc prefix for review.

Catalogue export requires Antelope's Python `antelope.datascope` bindings. Waveform indexing additionally requires `miniseed2db` on PATH. Ordinary pipeline/QuakeML steps remain usable without Antelope. The Python wrapper is structurally/mock tested; actual wfdisc reading must be validated in an Antelope installation before publication.

References: [Antelope CSS schema](https://anf.ucsd.edu/publications/css30.pdf) and [Antelope waveform indexing tutorial](https://www.jakewalter.net/antelopetutorial.html). Native command help for your installed Antelope version takes precedence.

## Metadata and scientific limits

Class labels and comments survive in event remarks and a membership sidecar. Candidate pick/origin membership is written as assoc with `timedef=n`; no residuals or historical location-use flags are fabricated. Unknown magnitude types remain null. Step 43/50 determine magnitude preference, including the provisional zero-as-missing interpretation.

wfdisc describes **segments in surviving triggered recordings**, not continuous acquisition. It inherits recorded MiniSEED timing; it does not correct historical clocks. Overlapping files and multiple events in one file remain possible. The optional inventory section delegates network-history tables to the chosen inventory provider; the native catalogue/wfdisc functions themselves do not synthesize them. Calibration fields are set to the schema's native null values by default because a verified counts-to-ground-motion calibration is not available.

The database references existing files. Moving the WAV archive can break links. Paths are resolved under `SEISAN_WAV_DB`, using `10_waveform_index.csv` by default; do not select SUDS files or REA S-files. No waveform sample files are written by this notebook. Output metadata lives beneath the configured external output root.

## 1. Configure the export

Use a fresh timestamped database prefix. The catalogue option defaults on; waveform indexing defaults off. For an index-only database, set `EXPORT_CATALOG=False` and `WRITE_WFDISC=True`. `MAX_WAVEFORM_FILES` is useful for a first trial; None selects all indexed paths. Run imports/definitions before executing the export cells.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys, json, math, pickle, shutil, subprocess
import pandas as pd
from obspy import UTCDateTime, read_events
from obspy.core.event import Catalog
from IPython.display import display
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Start in the repository root or 10_conversion.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths
CONFIG_FILE = None
P = pipeline_paths(config_file=CONFIG_FILE)
CATALOG_FILE = Path(P['CATALOG_PREFIX'] + '.xml')  # Canonical QuakeML; trusted .pkl also supported.
WAVEFORM_INDEX = Path(P['WAVEFORM_INDEX'])
SEISAN_WAV_ROOT = Path(P['SEISAN_WAV_DB'])
DATABASE_ROOT = Path(P['OUTPUT_ROOT']) / 'FAIR' / 'ANTELOPE'
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATABASE_PATH = DATABASE_ROOT / stamp / 'pinatubo'
EXPORT_CATALOG = True
WRITE_WFDISC = False
MAX_WAVEFORM_FILES = None
MINISEED2DB = 'miniseed2db'
COMMAND_TIMEOUT_SECONDS = 120
NULL_UNVERIFIED_CALIBRATION = True
EXPORT_INVENTORY_CSS = False
STATIONXML_FILE = Path(P['NETWORK_HISTORY_OUTPUT']) / 'pinatubo_provisional.xml'
INVENTORY_CSS_PREFIX = DATABASE_PATH.parent / 'network_history' / 'pinatubo_network'
NORMALIZE_CSS_SITECHAN_CONVENTIONS = True

print('Catalogue:', CATALOG_FILE)
print('Existing MiniSEED root:', SEISAN_WAV_ROOT)
print('Waveform index:', WAVEFORM_INDEX)
print('New CSS database:', DATABASE_PATH)

## 2. Catalogue exporter definitions

These definitions are independent of notebook 50 and do not run it. Native Antelope is imported only when export is called. A new database is required; existing prefixes are rejected. If a write fails, a partial database may remain for inspection—choose a new prefix on retry rather than appending blindly.

In [ ]:
from pathlib import Path
import json, math
import pandas as pd
from obspy import UTCDateTime

EVENT_CLASS_FLAGS = {
    'W_P_H': (True, True, True), 'W_P': (True, True, False),
    'W_H': (True, False, True), 'W_ONLY': (True, False, False),
    'P_H': (False, True, True), 'P_ONLY': (False, True, False), 'H_ONLY': (False, False, True),
}

def event_class_label(event):
    labels = [c.text.split(':', 1)[1] for c in event.comments if (c.text or '').startswith('event_class:')]
    if len(labels) != 1 or labels[0] not in EVENT_CLASS_FLAGS:
        raise ValueError(f'{event.resource_id}: missing/ambiguous event class')
    return labels[0]

def export_antelope_css3(catalog, dbpath):
    """Optional native Antelope catalogue export; no fabricated origins or wfdisc.

    Requires licensed Antelope Python bindings. Fixed widths/nulls are handled by
    Datascope, not a guessed SQL/CSV representation. Creates a NEW database only.
    """
    import json
    try:
        from antelope import datascope
    except ImportError as exc:
        raise ImportError('CSS3 export requires an Antelope-enabled Python environment; ordinary exports do not.') from exc
    dbpath = Path(dbpath)
    dbpath.parent.mkdir(parents=True, exist_ok=True)
    if dbpath.exists() or list(dbpath.parent.glob(dbpath.name + '.*')):
        raise FileExistsError(f'Refusing to overwrite CSS database: {dbpath}')
    # Validate class labels before creating files.
    for event in catalog:
        event_class_label(event)
    datascope.dbcreate(str(dbpath), 'css3.0')
    db = datascope.dbopen(str(dbpath), 'r+')
    manifest = []
    lddate = float(UTCDateTime())
    tables = {}
    def add(table, **values):
        pointer = tables.setdefault(table, db.lookup(table=table))
        pointer.addv(*tuple(values.items()))
    def remark(text):
        commid = db.nextid('commid')
        # CSS remark lines are at most 80 characters; no lost text.
        for lineno, start in enumerate(range(0, max(1, len(text)), 80), start=1):
            add('remark', commid=commid, lineno=lineno, remark=text[start:start+80], lddate=lddate)
        return commid
    try:
        for event in catalog:
            label = event_class_label(event)
            evid = db.nextid('evid')
            origin_ids = {}
            for origin in event.origins:
                if origin.time is None or origin.latitude is None or origin.longitude is None:
                    continue
                if not all(math.isfinite(float(v)) for v in [origin.latitude, origin.longitude]):
                    continue
                orid = db.nextid('orid')
                origin_ids[str(origin.resource_id)] = orid
                fields = dict(orid=orid, evid=evid, lat=float(origin.latitude), lon=float(origin.longitude),
                              time=float(origin.time), jdate=int(origin.time.strftime('%Y%j')),
                              auth='PINATUBO', lddate=lddate,
                              commid=remark('resource_id:' + str(origin.resource_id)))
                if origin.depth is not None and math.isfinite(float(origin.depth)):
                    fields['depth'] = float(origin.depth) / 1000
                add('origin', **fields)
            prefor = origin_ids.get(str(event.preferred_origin_id), next(iter(origin_ids.values()), -1))
            meta = dict(resource_id=str(event.resource_id), event_class=label,
                        waveform_present=EVENT_CLASS_FLAGS[label][0], picks_present=EVENT_CLASS_FLAGS[label][1],
                        hypocenter_present=EVENT_CLASS_FLAGS[label][2],
                        comments=[c.text for c in event.comments])
            add('event', evid=evid, prefor=prefor, auth='PINATUBO',
                commid=remark(json.dumps(meta, ensure_ascii=True)), lddate=lddate)
            pick_links = []
            for pick in event.picks:
                wid = pick.waveform_id
                if pick.time is None or wid is None or not wid.station_code:
                    raise ValueError(f'{pick.resource_id}: arrival lacks time/station; partial export retained for inspection')
                station = wid.station_code
                channel = wid.channel_code or '-'
                phase = pick.phase_hint or '-'
                # Reject unrepresentable IDs rather than silently truncate them.
                if len(station) > 6 or len(channel) > 8 or len(phase) > 8:
                    raise ValueError(f'CSS identifier too long: {station}/{channel}/{phase}')
                arid = db.nextid('arid')
                add('arrival', arid=arid, sta=station, chan=channel, iphase=phase,
                    time=float(pick.time), jdate=int(pick.time.strftime('%Y%j')), auth='PINATUBO',
                    commid=remark(json.dumps(dict(resource_id=str(pick.resource_id), seed_id=wid.get_seed_string()))), lddate=lddate)
                if prefor != -1:
                    add('assoc', arid=arid, orid=prefor, sta=station, phase=phase, timedef='n',
                        commid=remark('Association from catalogue event membership; not evidence of use in this location; residuals unknown.'), lddate=lddate)
                pick_links.append(dict(arid=arid, resource_id=str(pick.resource_id), seed_id=wid.get_seed_string()))
            for magnitude in event.magnitudes:
                if magnitude.mag is None or not math.isfinite(float(magnitude.mag)):
                    continue
                magtype = magnitude.magnitude_type or '-'
                if len(magtype) > 6:
                    raise ValueError(f'CSS magnitude type too long: {magtype}')
                add('netmag', magid=db.nextid('magid'), evid=evid,
                    orid=origin_ids.get(str(magnitude.origin_id), prefor), magtype=magtype,
                    magnitude=float(magnitude.mag), auth='PINATUBO',
                    commid=remark(json.dumps(dict(resource_id=str(magnitude.resource_id), comments=[c.text for c in magnitude.comments]))), lddate=lddate)
            manifest.append(dict(evid=evid, prefor=prefor, **meta, arrivals=pick_links,
                                 origin_id_map=origin_ids))
            dbpath.with_name(dbpath.name + '_membership.json').write_text(json.dumps(manifest, indent=2) + '\n')
    finally:
        db.close()
    dbpath.with_name(dbpath.name + '_README.txt').write_text(
        'Schema css3.0; native Antelope/Datascope writer. Catalogue metadata; optional wfdisc indexing is a separate cell in notebook 54.\n'
        'No fabricated origins: unlocated events have prefor=-1. Arrival/event membership for these is in the membership JSON.\n'
        'Assoc rows express candidate event membership, timedef=n; no residuals or claimed location use.\n'
        'The catalogue function itself writes no wfdisc or samples; notebook 54 can add native wfdisc indexing. No site/instrument/response export. Waveform filenames remain in comments/manifest.\n'
        'Unknown magnitude types remain CSS null; source values/provenance retained in netmag/remarks.\n')
    return dict(database=str(dbpath), events=len(manifest), membership=str(dbpath.with_name(dbpath.name + '_membership.json')))

## 3. Optional wfdisc indexing definitions

The wrapper invokes `miniseed2db FILE DATABASE` with an argument list, never a shell command. Processing one file at a time avoids command-length limits and provides file-specific stdout/stderr. It records input path/size/mtime and rejects files outside the configured WAV root. Input-stat changes are reported as errors. No station/channel/sample offsets are inferred from the event-index station lists.

An existing nonempty wfdisc is rejected to prevent duplicate append runs. Failed partial indexing must be reviewed; use a new database prefix for a clean rerun. Native successes/failures and resulting wfdisc fields are saved in CSVs. Native wfdisc rows retain calibration nulls unless you deliberately disable that policy after establishing calibration.

Files with native-command failures are logged and the loop continues. This is not an implicit successful export: inspect every error and the final row counts. No automatic resume, deduplication of overlapping recordings, waveform-event wftag association, or channel epoch reconstruction is performed. `miniseed2db` may create snetsta/schanloc mappings; inspect these with the native schema. A full SEED-ID header inventory remains the independent reference.

In [ ]:
def index_existing_miniseed(dbpath, waveform_index, wav_root, *, executable='miniseed2db',
                            max_files=None, timeout_seconds=120, null_calibration=True):
    import shutil, subprocess, json
    from antelope import datascope
    dbpath = Path(dbpath).resolve()
    wav_root = Path(wav_root).expanduser().resolve()
    if not dbpath.is_file():
        raise FileNotFoundError('Create the fresh catalogue/index-only database in section 4 first.')
    program = shutil.which(str(executable))
    if program is None:
        raise FileNotFoundError(f'{executable} is not on PATH; activate the Antelope environment.')
    index = pd.read_csv(waveform_index, low_memory=False)
    if 'wav_file' not in index:
        raise ValueError('Waveform index lacks wav_file.')
    paths = list(dict.fromkeys(str(Path(p).expanduser().resolve()) for p in index['wav_file'].dropna()))
    if max_files is not None:
        if max_files < 0:
            raise ValueError('max_files must be nonnegative or None.')
        paths = paths[:max_files]
    if not paths:
        raise ValueError('No selected waveform files.')
    db = datascope.dbopen(str(dbpath), 'r')
    try:
        wf = db.lookup(table='wfdisc')
        if wf.query(datascope.dbRECORD_COUNT):
            raise ValueError('wfdisc is already nonempty; use a fresh export to avoid duplicates.')
    finally:
        db.close()
    audit_path = dbpath.with_name(dbpath.name + '_wfdisc_index_audit.csv')
    import csv
    audit_columns = ['wav_file', 'status', 'returncode', 'stdout', 'stderr', 'size_bytes', 'mtime_ns', 'error']
    with audit_path.open('x', newline='') as handle:
        csv.DictWriter(handle, fieldnames=audit_columns).writeheader()
    rows = []
    for filename in paths:
        path = Path(filename)
        row = dict(wav_file=str(path), status='error')
        try:
            if not path.is_relative_to(wav_root):
                raise ValueError(f'Input is outside configured SEISAN WAV root: {path}')
            before = path.stat()
            if not path.is_file():
                raise ValueError('Input is not a regular waveform file.')
            completed = subprocess.run([program, str(path), str(dbpath)], capture_output=True,
                                       text=True, timeout=timeout_seconds, check=False,
                                       cwd=str(dbpath.parent))
            after = path.stat()
            row.update(returncode=completed.returncode, stdout=completed.stdout, stderr=completed.stderr,
                       size_bytes=before.st_size, mtime_ns=before.st_mtime_ns)
            if (after.st_size, after.st_mtime_ns) != (before.st_size, before.st_mtime_ns):
                raise RuntimeError('Source file size/mtime changed during indexing; investigate immediately.')
            row['status'] = 'indexed' if completed.returncode == 0 else 'native_command_failed'
        except Exception as exc:
            row['error'] = f'{type(exc).__name__}: {exc}'
        rows.append(row)
        with audit_path.open('a', newline='') as handle:
            csv.DictWriter(handle, fieldnames=audit_columns).writerow(row)
    db = datascope.dbopen(str(dbpath), 'r+')
    exported = []
    try:
        wf = db.lookup(table='wfdisc')
        count = wf.query(datascope.dbRECORD_COUNT)
        if null_calibration and count:
            wf.record = datascope.dbNULL
            calib_null, calper_null = wf.getv('calib', 'calper')
        fields = ['wfid', 'sta', 'chan', 'time', 'endtime', 'nsamp', 'samprate', 'datatype',
                  'dir', 'dfile', 'foff', 'calib', 'calper']
        for record in range(count):
            wf.record = record
            if null_calibration:
                wf.putv(('calib', calib_null), ('calper', calper_null))
            exported.append(dict(zip(fields, wf.getv(*fields))))
    finally:
        db.close()
    wfdisc_csv = dbpath.with_name(dbpath.name + '_wfdisc.csv')
    pd.DataFrame(exported, columns=fields).to_csv(wfdisc_csv, index=False)
    report = dict(selected_files=len(paths), command_successes=sum(r['status']=='indexed' for r in rows),
                  errors=sum(r['status']!='indexed' for r in rows), wfdisc_rows=len(exported),
                  calibration_policy='native_schema_nulls' if null_calibration else 'native_indexer_defaults_unverified',
                  audit_csv=str(audit_path), wfdisc_csv=str(wfdisc_csv),
                  validation='Native indexing completed; waveform decoding/readback in Antelope still requires review.')
    dbpath.with_name(dbpath.name + '_wfdisc_summary.json').write_text(json.dumps(report, indent=2) + '\n')
    return report

## 4. Create the fresh CSS database

This is the first writing cell. Catalogue export defaults on. Turning it off creates an empty CSS database for optional waveform-only indexing. No QuakeML assembly or source conversion is rerun. Running this cell twice with the same prefix is rejected; rerun the configuration cell to choose a new timestamped prefix.

In [ ]:
if EXPORT_CATALOG:
    if CATALOG_FILE.suffix.lower() in ('.pkl', '.pickle'):
        with CATALOG_FILE.open('rb') as handle:
            catalog = pickle.load(handle)  # Trusted local pipeline pickle only.
    else:
        catalog = read_events(str(CATALOG_FILE))
else:
    catalog = Catalog()
css_result = export_antelope_css3(catalog, DATABASE_PATH)
display(css_result)

## 5. Optionally wrap existing WAV MiniSEED in wfdisc

Set WRITE_WFDISC=True in the configuration cell before running this cell. The selected inputs are the existing MiniSEED paths from step 10, including waveforms without picks/hypocenters. A catalogue subset does not restrict waveform indexing. Consider MAX_WAVEFORM_FILES=2 for a first native readback trial, then create a fresh full export after validation.

In [ ]:
if WRITE_WFDISC:
    wfdisc_result = index_existing_miniseed(
        DATABASE_PATH, WAVEFORM_INDEX, SEISAN_WAV_ROOT,
        executable=MINISEED2DB, max_files=MAX_WAVEFORM_FILES,
        timeout_seconds=COMMAND_TIMEOUT_SECONDS,
        null_calibration=NULL_UNVERIFIED_CALIBRATION)
    display(wfdisc_result)
else:
    print('wfdisc indexing disabled; existing MiniSEED files remain untouched.')

## 6. Optional: step-15 StationXML → CSS network-history tables

Set EXPORT_INVENTORY_CSS=True to use **ObsPy `Inventory.write(prefix, format="CSS")`**. This section does not require Antelope bindings and can run after configuration/definitions without executing the catalogue or wfdisc cells. The default input is the documentary `pinatubo_provisional.xml` from step 15; choose another audited product explicitly if needed.

ObsPy writes affiliation, network, site, sitechan and optional remark tables. It does **not** export response objects, sensor/instrument tables or complete response history. The exact input StationXML is copied alongside them, and the JSON audit retains full network/station/location/channel IDs, channel coordinates, exact epoch bounds, orientation and response-presence flags. Do not substitute guessed gain/calibration values.

CSS day-based ondate/offdate fields lose sub-day precision. Response-only changes and multiple same-day channel epochs can produce repeated CSS keys; the audit counts them. Location codes and channel-specific coordinates are not fully represented by the stock writer. Review the staged tables before native integration. This export is not a lossless replacement for StationXML.

The installed ObsPy 1.5 writer emits channel depth in metres and dip measured from horizontal directly into sitechan. With NORMALIZE_CSS_SITECHAN_CONVENTIONS=True, the wrapper detects that implementation and converts edepth to kilometres and vang to angle from vertical (90 + dip). Original writer rows are retained in obspy_raw/sitechan.txt; each adjustment is logged. Future writer implementations are checked before corrections, avoiding double application. These metadata transformations do not supply instrument responses.

Tables use a fresh INVENTORY_CSS_PREFIX under a separate network_history staging folder. This prevents overwriting catalogue remark records or native waveform ID-mapping tables. Do not copy table files over the main database: review epoch keys and channel mappings, then use an explicit native merge/linking workflow. No automatic merge or wfdisc chanid reassignment is performed here.

Reference: [ObsPy CSS writer documentation](https://docs.obspy.org/packages/autogen/obspy.io.css.station._write_css.html).


In [ ]:
def export_stationxml_css(stationxml_file, output_prefix, *, normalize_sitechan=True):
    """Use Inventory.write(CSS), retaining full epoch provenance and response source."""
    import hashlib, json, shutil, inspect
    from obspy import read_inventory
    from obspy.io.css import station as css_station
    source = Path(stationxml_file)
    prefix = Path(output_prefix)
    if not source.is_file():
        raise FileNotFoundError(f'Run step 15 or select its StationXML explicitly: {source}')
    if prefix.parent.exists():
        raise FileExistsError(f'Choose a fresh inventory staging folder: {prefix.parent}')
    inventory = read_inventory(str(source))
    epochs = []
    warnings = []
    for network in inventory:
        for station in network:
            for channel in station:
                if len(station.code) > 6 or len(channel.code) > 8:
                    raise ValueError('Station/channel identifier exceeds CSS width; supply an explicit mapping first.')
                epochs.append(dict(network=network.code, station=station.code, location=channel.location_code,
                    channel=channel.code, start=str(channel.start_date) if channel.start_date else None,
                    end=str(channel.end_date) if channel.end_date else None,
                    latitude=channel.latitude, longitude=channel.longitude, elevation_m=channel.elevation,
                    depth_m=channel.depth, azimuth=channel.azimuth, dip=channel.dip,
                    sample_rate=channel.sample_rate, has_response=channel.response is not None))
                if channel.location_code:
                    warnings.append('Location codes are retained in the sidecar but not represented by this CSS writer.')
    prefix.parent.mkdir(parents=True, exist_ok=False)
    audit_path = prefix.with_name(prefix.name + '_stationxml_provenance.json')
    audit = dict(source_stationxml=str(source.resolve()), source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),
        output_prefix=str(prefix), channel_epochs=epochs, response_exported=False,
        warnings=warnings, conversion='ObsPy Inventory.write(format=CSS); network metadata only')
    try:
        inventory.write(str(prefix), format='CSS')
        # Preserve the exact source with responses and sub-day epochs.
        shutil.copy2(source, prefix.with_name(prefix.name + '_source.xml'))
        sitechan = prefix.with_suffix('.sitechan')
        adjustments = []
        if normalize_sitechan and sitechan.is_file():
            lines = sitechan.read_text().splitlines()
            if len(lines) != len(epochs):
                raise ValueError('Unexpected CSS sitechan row count; no automatic convention adjustment applied.')
            # ObsPy 1.5 writer uses cha.depth (metres) and cha.dip directly.
            # CSS edepth is km; vang is measured from vertical, hence 90 + dip.
            # Detect this exact implementation, so a future fixed writer is not double-corrected.
            writer_source = inspect.getsource(css_station._write_css)
            old_depth = 'cha.depth,' in writer_source and 'cha.depth /' not in writer_source
            old_angle = "(cha.dip, )" in writer_source
            rawdir = prefix.parent / 'obspy_raw'
            rawdir.mkdir(exist_ok=True)
            (rawdir / 'sitechan.txt').write_text('\n'.join(lines) + '\n')
            for i, (line, epoch) in enumerate(zip(lines, epochs)):
                # Field boundaries follow the verified ObsPy CSS writer format.
                if line[:6].strip() != epoch['station'] or line[7:15].strip() != epoch['channel']:
                    raise ValueError('Unexpected sitechan field layout; preserve source and review manually.')
                if old_depth:
                    line = line[:48] + f"{epoch['depth_m'] / 1000:9.4f}" + line[57:]
                if old_angle and epoch['dip'] is not None:
                    line = line[:65] + f"{90 + epoch['dip']:6.1f}" + line[71:]
                if old_depth or old_angle:
                    adjustments.append(dict(row=i+1, depth_m_to_km=old_depth,
                                            dip_to_angle_from_vertical=old_angle and epoch['dip'] is not None))
                lines[i] = line
            sitechan.write_text('\n'.join(lines) + '\n')
        audit['sitechan_convention_adjustments'] = adjustments
        if sitechan.is_file():
            keys = [(line[:6].strip(), line[7:15].strip(), line[16:24].strip()) for line in sitechan.read_text().splitlines()]
            duplicates = len(keys) - len(set(keys))
            audit['duplicate_sitechan_day_keys'] = duplicates
            if duplicates:
                warnings.append(f'{duplicates} repeated station/channel/ondate keys: sub-day or response-only epochs need reconciliation before native merge.')
        warnings.append('CSS day-based epochs cannot preserve exact sub-day boundary precision; use the StationXML and sidecar.')
        warnings.append('Response objects, instrument/sensor tables and full location/response history are not exported by Inventory.write(CSS).')
        audit['status'] = 'exported_to_separate_staging_prefix'
    except Exception as exc:
        audit.update(status='failed', error=f'{type(exc).__name__}: {exc}')
        raise
    finally:
        audit['generated_files'] = [str(path) for path in sorted(prefix.parent.rglob('*')) if path.is_file() and path != audit_path]
        audit_path.write_text(json.dumps(audit, indent=2, default=str) + '\n')
    return audit


In [ ]:
if EXPORT_INVENTORY_CSS:
    inventory_css_result = export_stationxml_css(
        STATIONXML_FILE, INVENTORY_CSS_PREFIX,
        normalize_sitechan=NORMALIZE_CSS_SITECHAN_CONVENTIONS)
    display(inventory_css_result)
else:
    print('StationXML-to-CSS network metadata export disabled.')


## 7. Native validation before using the export

Inspect the per-file command log, error count, total wfdisc rows and full input/output paths. A zero command return code alone does not prove every trace was indexed. In Antelope, open the database with dbe/dbpick and read representative waveforms, including a multichannel event, partial recovery, overlapping segments and a file with multiple pick groups. Compare station/channel mapping, sample counts/rates, starts/ends and decoded samples against ObsPy and the original WAV file. Confirm references continue to point to those existing files rather than unintended copies.

Compare EVENT_CLASS remarks/membership with the canonical catalogue manifest. Check that unlocated records have no invented origins and calibration is null. This notebook does not provide fake station locations or complete responses to make a viewer appear calibrated. Record native validation results with the export; rerun only into a fresh prefix when source/index versions change.